# Baseline B1 - generation

Author: Mathias Marryat Husted

This notebook provides a reproducible way to generate the baseline data sheets. Checks are performed at each step to ensure data consistency.

In [15]:
# Imports and constants
import pandas as pd
import gzip


BENCHMARK_PATH = "../../data/dataset/benchmark.csv"
CONTROL_PATH = "../../data/dataset/control.csv"
# https://www.genenames.org/download/
HGNC_PATH = "../../data/reference/hgnc_complete_set.txt"
# https://ftp.ncbi.nlm.nih.gov/pubchem/Compound/Extras/
PUBCHEM_DIR = "../../data/reference"

# Initialize all the data in pandas
# Read everything as as string and turn off NA coercion
benchmark = pd.read_csv(BENCHMARK_PATH, sep=",", dtype=str, keep_default_na=False)
control = pd.read_csv(CONTROL_PATH, sep=",", dtype=str, keep_default_na=False)
hgnc_data = pd.read_csv(HGNC_PATH, sep="\t", dtype=str, keep_default_na=False)


This script performs the baseline runs B1, B2, B3, B4 taking the benchmark.csv or similar data set as the input. The data sets mentioned above must be present in the directories


In [16]:
# Double check gene coverage of HGNC symbol == Control canonical_name
def coverage_check(hgnc_data, control):
    control_genes = control[control["entity_type"] == "gene"]
    merged_df = pd.merge(
        left=control_genes,
        right=hgnc_data,
        left_on="hgnc_id",
        right_on="hgnc_id",
        how="left",
        indicator=True,
    )

    not_covered = merged_df[merged_df["_merge"] == "left_only"]
    assert len(not_covered) == 0, f"{len(not_covered)} IDs may be non-existent!"
    mismatched = merged_df[merged_df["symbol"] != merged_df["canonical_name"]]
    assert len(mismatched) == 0, f"The HGNC symbol does not match the canonical name on {len(mismatched)} rows!"
    assert len(merged_df) == len(control_genes), "Row count changed after merge"
    print(f"Gene coverage check {"passed" if len(not_covered) == 0 else "failed"}, {len(merged_df)} total items")
    return merged_df

merged = coverage_check(hgnc_data, control)

Gene coverage check passed, 80 total items


# B1: Exact lookup against DB lookup
Requires: Database download of both HGNC as well as PubChem

In [ ]:
# Claude Opus 5.5 rewrote this cell based on a previous cell (23.09.2026)
# Prompt: Build a function that performs the lookup of the gene synonyms according to the given data format,
# comparing them to symbol, prev_symbol and alias_symbol.
# Inputs:
# - Prompt (as written below)
# - This file (current state: File ends after the saving of the b1b_chem_synonym csv file is created. Includes run_b1_genes)
# - Fallback order
# -> Claude copied the run_b1_genes function and fit its code into the logic
# - hgnc_complete_set.txt
# - benchmark.csv
# - control.csv
"""

"""
# 
FIELD_ORDER = ["symbol", "prev_symbol", "alias_symbol", "name", "prev_name", "alias_name"]  # Fallback order, extended to names

# One row per (gene, name, field): the gene synonym vocabulary in long format
def build_gene_synonym_table(hgnc_data):
    approved = hgnc_data[(hgnc_data["status"] == "Approved") & (hgnc_data["symbol"] != "")]
    # Approved symbols are unique, so this is a plain 1:1 lookup
    assert approved["symbol"].is_unique, "duplicate approved symbols in the HGNC data"
    parts = []
    for field in FIELD_ORDER:
        part = approved[["hgnc_id", field]].rename(columns={field: "name"})
        part["name"] = part["name"].str.split("|")   # pipe-separated lists -> Python lists
        part = part.explode("name")                   # one row per name
        part = part[part["name"] != ""]               # genes with an empty field
        part["field"] = field
        parts.append(part)
    table = pd.concat(parts, ignore_index=True)
    table["folded"] = table["name"].str.casefold()
    return table
"""
THE FUNCTION CLAUDE WAS ASKED TO MERGE BACK INTO THE RUN FUNCTION:
# Run B1 on genes, synonym tier: symbol -> prev_symbol -> alias_symbol, first matching field wins
def run_b1_genes_synonym(synonym_table, benchmark, fold=False):
    benchmark = benchmark[benchmark["entity_type"] == "gene"]  # gold entity type used for routing
    results = []
    for row in benchmark.itertuples():
        if fold:
            matches = synonym_table[synonym_table["folded"] == row.input.casefold()]
        else:
            matches = synonym_table[synonym_table["name"] == row.input]
        matched_field = None
        for field in FIELD_ORDER:
            if (matches["field"] == field).any():
                matches = matches[matches["field"] == field]  # keep only the first field that matched
                matched_field = field
                break
        n_cand = matches["hgnc_id"].nunique()  # distinct genes within the winning field
        returned_id = matches["hgnc_id"].iloc[0] if n_cand == 1 else None
        results.append({
            "id": row.id,
            "canonical_name": row.canonical_name,
            "input": row.input,
            "n_candidates": n_cand,
            "noise_type": row.noise_type,
            "id_status": row.id_status,
            "accepted_id": row.accepted_id,
            "returned_id": returned_id,
            "matched_field": matched_field,
        })
    return pd.DataFrame(results)
"""
# Check proper pipe separation and no stray quotes
hgnc_data.loc[hgnc_data["prev_name"] != "", ["symbol", "name", "prev_name", "alias_name"]].head(10)

,symbol,name,prev_name,alias_name
1,A1BG-AS1,A1BG antisense RNA 1,non-protein coding RNA 181|A1BG antisense RNA ...,
4,A2M-AS1,A2M antisense RNA 1,A2M antisense RNA 1 (non-protein coding),
5,A2ML1,alpha-2-macroglobulin like 1,"C3 and PZP-like, alpha-2-macroglobulin domain ...",
6,A2ML1-AS1,A2ML1 antisense RNA 1,A2ML1 antisense RNA 1 (non-protein coding),
7,A2ML1-AS2,A2ML1 antisense RNA 2,A2ML1 antisense RNA 2 (non-protein coding),
8,A3GALT2,"alpha 1,3-galactosyltransferase 2","alpha 1,3-galactosyltransferase 2, pseudogene",iGb3 synthase|isoglobotriaosylceramide synthase
9,A4GALT,"alpha 1,4-galactosyltransferase (P1PK blood gr...","alpha 1,4-galactosyltransferase (globotriaosyl...",Gb3 synthase|CD77 synthase|globotriaosylcerami...
11,AAAS,aladin WD repeat nucleoporin,"achalasia, adrenocortical insufficiency, alacr...","aladin|Allgrove, triple-A|adracalin"
13,AACSP1,acetoacetyl-CoA synthetase pseudogene 1,acetoacetyl-CoA synthetase-like,
14,AADAC,arylacetamide deacetylase,arylacetamide deacetylase (esterase),


In [ ]:
"""
Pre-Claude function:
# Run the benchmark on genes
def run_b1a(hgnc_data, benchmark):
    # Read only approved symbols
    hgnc_data = hgnc_data[hgnc_data["status"] == "Approved"]
    hgnc_data = hgnc_data[hgnc_data["symbol"] != ""]
    benchmark = benchmark[benchmark["entity_type"] == "gene"] # Gold entity type used for the benchmark
    # Approved symbols are unique, so this is a plain 1:1 lookup
    assert hgnc_data["symbol"].is_unique, "duplicate approved symbols in the HGNC data"
    symbol_index = hgnc_data.set_index("symbol")["hgnc_id"].to_dict()
    print(f"{HGNC_PATH}: {len(hgnc_data)} approved records, {len(symbol_index)} symbols")
    
    # Look up gene data
    results = []
    for row in benchmark.itertuples():
        # The actual B1 baseline check: exact lookup of the raw, noisy input
        hgnc_id = symbol_index.get(row.input)

        results.append({
            "id": row.id,
            "canonical_name": row.canonical_name,
            "input": row.input,
            "n_candidates": int(hgnc_id is not None),
            "noise_type": row.noise_type,
            "id_status": row.id_status,
            "accepted_id": row.accepted_id,
            "returned_id": hgnc_id,
        })

    return pd.DataFrame(results)
"""

'\nPre-Claude function:\n# Run the benchmark on genes\ndef run_b1a(hgnc_data, benchmark):\n    # Read only approved symbols\n    hgnc_data = hgnc_data[hgnc_data["status"] == "Approved"]\n    hgnc_data = hgnc_data[hgnc_data["symbol"] != ""]\n    benchmark = benchmark[benchmark["entity_type"] == "gene"] # Gold entity type used for the benchmark\n    # Approved symbols are unique, so this is a plain 1:1 lookup\n    assert hgnc_data["symbol"].is_unique, "duplicate approved symbols in the HGNC data"\n    symbol_index = hgnc_data.set_index("symbol")["hgnc_id"].to_dict()\n    print(f"{HGNC_PATH}: {len(hgnc_data)} approved records, {len(symbol_index)} symbols")\n\n    # Look up gene data\n    results = []\n    for row in benchmark.itertuples():\n        # The actual B1 baseline check: exact lookup of the raw, noisy input\n        hgnc_id = symbol_index.get(row.input)\n\n        results.append({\n            "id": row.id,\n            "canonical_name": row.canonical_name,\n            "input": 

In [19]:
# Run the benchmark on genes
def run_b1_genes(synonym_table, benchmark, fold=False): # Claude: hgnc_data -> synonym_table
    benchmark = benchmark[benchmark["entity_type"] == "gene"] # Gold entity type used for the benchmark
    # Look up gene data
    results = []
    for row in benchmark.itertuples():
        # The B1 baseline check: exact lookup of the raw, noisy input
        if fold:
            matches = synonym_table[synonym_table["folded"] == row.input.casefold()]
        else:
            matches = synonym_table[synonym_table["name"] == row.input]
        matched_field = None
        for field in FIELD_ORDER: # Claude: added this for loop
            if (matches["field"] == field).any():
                matches = matches[matches["field"] == field]  # keep only the first field that matched
                matched_field = field
                break
        n_cand = matches["hgnc_id"].nunique()  # Is the ID unique?
        returned_id = matches["hgnc_id"].iloc[0] if n_cand == 1 else None  # Return the matched ID if it's unique
        results.append({
            "id": row.id,
            "canonical_name": row.canonical_name,
            "input": row.input,
            "n_candidates": n_cand,
            "noise_type": row.noise_type,
            "id_status": row.id_status,
            "accepted_id": row.accepted_id,
            "returned_id": returned_id,
            "matched_field": matched_field # Claude: Added this data point
        })

    return pd.DataFrame(results)

In [20]:
coverage_check(hgnc_data, control)

syn_table = build_gene_synonym_table(hgnc_data)
canonical = syn_table[syn_table["field"] == "symbol"]

control_genes = control.assign(noise_type="control")
for fold in [False, True]: #B1a (byte-exact) and B1b (case-folded)
    # Positive control using the control data set against the benchmark (canonical column of the syn_table)
    b1_control_results = run_b1_genes(canonical, control_genes, fold)
    assert (b1_control_results['returned_id'] == b1_control_results['accepted_id']).sum() == len(b1_control_results)
    print(f"Preliminary control run with fold={fold}. Expect 80/80")
    print(f"{(b1_control_results['returned_id'] == b1_control_results['accepted_id']).sum()}/{len(b1_control_results)}")
    a_or_b = "b" if fold else "a"
    # The actual results
    # CANONICAL TIER
    b1_results_canonical = run_b1_genes(canonical, benchmark, fold)
    save_path_c = f"../../results/baselines/b1{a_or_b}_gene_canonical.csv"
    b1_results_canonical.to_csv(save_path_c, index=False)
    # SYNONYM TIER
    b1_results_synonym = run_b1_genes(syn_table, benchmark, fold)
    save_path_s = f"../../results/baselines/b1{a_or_b}_gene_synonym.csv"
    b1_results_synonym.to_csv(save_path_s, index=False)



Gene coverage check passed, 80 total items
Preliminary control run with fold=False. Expect 80/80
80/80
Preliminary control run with fold=True. Expect 80/80
80/80


Change: Add case folding because the synonym file stores some chemicals in all-caps, others capitalized etc.

- Chloroform -> CHLOROFORM
- Melatonin -> Melatonin
- Hexane -> HEXANE

In [21]:
def read_pubchem(path, search_names, gold_cids):
    name_matches = []
    gold_rows = []
    invalid_lines = 0
    folded_names = {n.casefold() for n in search_names}
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            elements = line.rstrip("\n").split("\t", 1)
            if len(elements) != 2:
                invalid_lines += 1
                continue
            cid, name = elements
            if name.casefold() in folded_names:
                name_matches.append((cid, name))
            if cid in gold_cids:
                gold_rows.append((cid, name))
    print(f"Number of invalid lines skipped: {invalid_lines}")
    return (pd.DataFrame(name_matches, columns=["cid", "name"]),
            pd.DataFrame(gold_rows, columns=["cid", "name"]))

benchmark_chemicals = benchmark[benchmark["entity_type"] == "chemical"]
control_chemicals = control[control["entity_type"] == "chemical"]

# Search names are all unique inputs across the union of the benchmark and control input columns (for control, it's equal to canonical_name)
search_names = set(benchmark_chemicals["input"]) | set(control_chemicals["canonical_name"])


print(f"# chemicals in benchmark: {len(benchmark_chemicals)}, # chemicals with a unique name in the union: {len(search_names)}")


gold_cids = set(control_chemicals["pubchem_id"]) - {""}

print(f"# chemicals in control: {len(control_chemicals)}, # chemicals with a PubChem ID: {len(gold_cids)}")

# chemicals in benchmark: 162, # chemicals with a unique name in the union: 264
# chemicals in control: 109, # chemicals with a PubChem ID: 108


In [22]:
# Execute the read_pubchem function for the CID-Title.gz file. Match once on title (from the input chemicals) and once on
# the pre-verified PubChem IDs (expect all to match)
# Write the result to a CSV for later reference
title_hits, title_gold = read_pubchem(f"{PUBCHEM_DIR}/CID-Title.gz", search_names, gold_cids)

title_hits.to_csv("../../results/intermediate/title_hits.csv", index=False)
title_gold.to_csv("../../results/intermediate/title_gold.csv", index=False)

Number of invalid lines skipped: 0


In [23]:
# Now we'll do the same but for the filtered synonym file
synonym_hits, synonym_gold = read_pubchem(f"{PUBCHEM_DIR}/CID-Synonym-filtered.gz", search_names, gold_cids)

synonym_hits.to_csv("../../results/intermediate/synonym_hits.csv", index=False)
synonym_gold.to_csv("../../results/intermediate/synonym_gold.csv", index=False)

Number of invalid lines skipped: 0


Analyze the data from our function. Make sure you've generated all the CSVs before continuing here

In [24]:
def run_b1_chemicals(hits_file, benchmark, fold=False):
    data = pd.read_csv(hits_file, sep=",", dtype=str, keep_default_na=False)
    data["folded"] = data["name"].str.casefold()
    benchmark = benchmark[benchmark["entity_type"] == "chemical"]
    results = []
    for row in benchmark.itertuples():
        if fold:
            matches = data[data["folded"] == row.input.casefold()]
        else:
            matches = data[data["name"] == row.input]
        n_cand = matches["cid"].nunique() # Is the ID unique?
        returned_id = matches["cid"].iloc[0] if n_cand == 1 else None # Return the matched ID if it's unique
        results.append({
            "id": row.id,
            "canonical_name": row.canonical_name,
            "input": row.input,
            "n_candidates": n_cand,
            "noise_type": row.noise_type,
            "id_status": row.id_status,
            "accepted_id": row.accepted_id,
            "returned_id": returned_id,
        })
    return pd.DataFrame(results)

In [25]:
# Find chemicals that match directly over their canonical name WITHOUT folding (case-sensitive)
b1a_chem_canonical = run_b1_chemicals("../../results/intermediate/title_hits.csv", benchmark, False)
b1a_chem_canonical.to_csv("../../results/baselines/b1a_chem_canonical.csv", index=False)
print(f"B1a (byte-based, canonical): Number of chemicals with exactly 1 ID: {b1a_chem_canonical["returned_id"].notna().sum()}/{b1a_chem_canonical.shape[0]}")
# Find chemicals that match directly over their canonical name WITH folding (case-insensitive)
b1b_chem_canonical = run_b1_chemicals("../../results/intermediate/title_hits.csv", benchmark, True)
b1b_chem_canonical.to_csv("../../results/baselines/b1b_chem_canonical.csv", index=False)
print(f"B1b (case-folded, canonical): Number of chemicals with exactly 1 ID: {b1b_chem_canonical["returned_id"].notna().sum()}/{b1b_chem_canonical.shape[0]}")


B1a (byte-based, canonical): Number of chemicals with exactly 1 ID: 3/162
B1b (case-folded, canonical): Number of chemicals with exactly 1 ID: 22/162


In [26]:
# Find chemicals that match over their canonical name OR a synonym thereof
b1a_chem_synonym = run_b1_chemicals("../../results/intermediate/synonym_hits.csv", benchmark, False)
b1a_chem_synonym.to_csv("../../results/baselines/b1a_chem_synonym.csv", index=False)
print(f"B1a (byte-based, synonym):Number of chemicals with exactly 1 ID: {b1a_chem_synonym["returned_id"].notna().sum()}/{b1a_chem_synonym.shape[0]}")

b1b_chem_synonym = run_b1_chemicals("../../results/intermediate/synonym_hits.csv", benchmark, True)
b1b_chem_synonym.to_csv("../../results/baselines/b1b_chem_synonym.csv", index=False)
print(f"B1b: Number of chemicals with exactly 1 ID: {b1b_chem_synonym["returned_id"].notna().sum()}/{b1b_chem_synonym.shape[0]}")

B1a (byte-based, synonym):Number of chemicals with exactly 1 ID: 44/162
B1b: Number of chemicals with exactly 1 ID: 94/162
